# **RQ1**: Number of neighbors per node over time

Costruisco il grafo con igraph, con attributi di token, peso archi e freq parola

In [7]:
import pandas as pd
import igraph as ig
from pathlib import Path
import time

edges_dir = Path("../edges_analysis/edges")
vec_dir = Path("../embedding/word_embeddings_cleaned")
freq_dir = Path("../embedding/freq snap"
)

output_dir = Path("igraph_graphs")
output_dir.mkdir(exist_ok=True)

percentile = 99

print("[START] Costruzione grafi igraph con freq nodo")

for snap in range(1, 11):
    t0 = time.time()

    edges_tsv = edges_dir / f"edges_snap{snap}_p{percentile}.tsv"
    vec_file = vec_dir / f"fasttext_snap{snap}_filt.vec"
    freq_file = freq_dir / f"vocab_freq_snap{snap}.txt"
    out_graph = output_dir / f"graph_snap{snap}_p{percentile}.igp"

    print(f"\n[SNAPSHOT {snap}]")

    # ---------- ARCHI ----------
    df_edges = pd.read_csv(edges_tsv, sep="\t")

    src = df_edges.iloc[:, 0].astype(int).to_numpy()
    dst = df_edges.iloc[:, 1].astype(int).to_numpy()

    has_weight = df_edges.shape[1] > 2
    if has_weight:
        weights = df_edges.iloc[:, 2].astype(float).to_numpy()

    # ---------- TOKEN ----------
    tokens = []
    with open(vec_file, encoding="utf-8") as f:
        next(f)
        for line in f:
            tokens.append(line.split()[0])

    n_nodes = len(tokens)

    # ---------- FREQUENZE ----------
    freq_map = {}
    with open(freq_file, encoding="utf-8") as f:
        for line in f:
            token, freq = line.split()
            freq_map[token] = int(freq)

    # allineamento token → freq (0 se assente)
    freqs = [freq_map.get(tok, 0) for tok in tokens]

    # ---------- GRAFO ----------
    g = ig.Graph(n=n_nodes, directed=False)

    g.vs["token"] = tokens
    g.vs["freq"] = freqs

    g.add_edges(zip(src, dst))

    if has_weight:
        g.es["weight"] = weights

    # ---------- SALVATAGGIO ----------
    g.write_pickle(out_graph)

    print(
        f"[DONE] nodi={n_nodes}, archi={g.ecount()}, "
        f"freq=yes, peso={'sì' if has_weight else 'no'} "
        f"in {time.time() - t0:.2f}s"
    )

print("\n[TOTAL] Tutti i grafi salvati")


[START] Costruzione grafi igraph con freq nodo

[SNAPSHOT 1]
[DONE] nodi=49460, archi=12169053, freq=yes, peso=sì in 108.49s

[SNAPSHOT 2]
[DONE] nodi=47189, archi=11035778, freq=yes, peso=sì in 98.11s

[SNAPSHOT 3]
[DONE] nodi=64187, archi=20314009, freq=yes, peso=sì in 211.99s

[SNAPSHOT 4]
[DONE] nodi=64631, archi=21038711, freq=yes, peso=sì in 157.21s

[SNAPSHOT 5]
[DONE] nodi=75488, archi=28498747, freq=yes, peso=sì in 243.09s

[SNAPSHOT 6]
[DONE] nodi=82826, archi=34398376, freq=yes, peso=sì in 254.09s

[SNAPSHOT 7]
[DONE] nodi=64187, archi=20804061, freq=yes, peso=sì in 84.53s

[SNAPSHOT 8]
[DONE] nodi=58410, archi=17188815, freq=yes, peso=sì in 66.42s

[SNAPSHOT 9]
[DONE] nodi=52701, archi=14015080, freq=yes, peso=sì in 52.85s

[SNAPSHOT 10]
[DONE] nodi=48872, archi=11964184, freq=yes, peso=sì in 64.42s

[TOTAL] Tutti i grafi salvati


Per ogni nodo in ogni snapshot calcolo e scrivo su un file csv il grado.

In [10]:
import pandas as pd
import igraph as ig
from pathlib import Path
import time

graph_dir = Path("igraph_graphs")        
output_dir = Path("degree_csv_igraph_snapshot")
output_dir.mkdir(exist_ok=True)

percentile = 99

print("[START] Estrazione degree dai grafi igraph")

for snap in range(1, 11):
    t0 = time.time()

    graph_file = graph_dir / f"graph_snap{snap}_p{percentile}.igp"
    output_csv = output_dir / f"node_degree_snap{snap}_p{percentile}.csv"

    print(f"\n[SNAPSHOT {snap}]")

    # importo g
    g = ig.Graph.Read_Pickle(graph_file)

   
    degrees = g.degree()   # numero di vicini

   
    tokens = g.vs["token"]

    df_out = pd.DataFrame({
        "token": tokens,
        "degree": degrees
    })

    df_out.to_csv(output_csv, index=False)

    print(
        f"[DONE] nodi={g.vcount()}, archi={g.ecount()} "
        f"in {time.time() - t0:.2f}s"
    )

print("\n[TOTAL] Tutti i CSV generati")


[START] Estrazione degree dai grafi igraph

[SNAPSHOT 1]
[DONE] nodi=49460, archi=12169053 in 9.65s

[SNAPSHOT 2]
[DONE] nodi=47189, archi=11035778 in 8.31s

[SNAPSHOT 3]
[DONE] nodi=64187, archi=20314009 in 22.74s

[SNAPSHOT 4]
[DONE] nodi=64631, archi=21038711 in 33.90s

[SNAPSHOT 5]
[DONE] nodi=75488, archi=28498747 in 79.16s

[SNAPSHOT 6]
[DONE] nodi=82826, archi=34398376 in 122.43s

[SNAPSHOT 7]
[DONE] nodi=64187, archi=20804061 in 40.31s

[SNAPSHOT 8]
[DONE] nodi=58410, archi=17188815 in 17.78s

[SNAPSHOT 9]
[DONE] nodi=52701, archi=14015080 in 12.08s

[SNAPSHOT 10]
[DONE] nodi=48872, archi=11964184 in 8.77s

[TOTAL] Tutti i CSV generati
